In [1]:
# ============================================================
# SCRIPT 1: CRUCE LIMPIO SAIC / CENSOS + DENUE
# Proyecto: Nearshoring_Project
#
# Objetivo:
# Crear una base integrada municipal para 2018 y 2023
# sin crear todavía ratios, scores adicionales o tipologías.
#
# Inputs:
# - data/processed/municipality_industrial_capacity_panel.csv
# - data/processed/municipality_denue_support_summary.csv
#
# Output principal:
# - data/processed/municipality_industrial_b2b_opportunity_panel_raw.csv
#
# Unidad:
# year × municipality
# ============================================================


# ------------------------------------------------------------
# 0. Montar Google Drive
# ------------------------------------------------------------

from google.colab import drive
drive.mount('/content/drive')


# ------------------------------------------------------------
# 1. Librerías
# ------------------------------------------------------------

import pandas as pd
import numpy as np
from pathlib import Path


# ------------------------------------------------------------
# 2. Rutas
# ------------------------------------------------------------

project_path = Path("/content/drive/MyDrive/Nearshoring_Project")

processed_path = project_path / "data" / "processed"
tables_path = project_path / "outputs" / "tables"

tables_path.mkdir(parents=True, exist_ok=True)

saic_path = processed_path / "municipality_industrial_capacity_panel.csv"
denue_path = processed_path / "municipality_denue_support_summary.csv"

output_path = processed_path / "municipality_industrial_b2b_opportunity_panel_raw.csv"
validation_path = tables_path / "industrial_b2b_merge_validation_raw.csv"
sheets_output_path = tables_path / "industrial_b2b_opportunity_panel_raw_for_google_sheets.csv"


# ------------------------------------------------------------
# 3. Verificar archivos
# ------------------------------------------------------------

print("VERIFICACIÓN DE ARCHIVOS")
print("=" * 100)

print("SAIC / Censos:")
print(saic_path)
print("OK" if saic_path.exists() else "NO ENCONTRADO")

print("\nDENUE:")
print(denue_path)
print("OK" if denue_path.exists() else "NO ENCONTRADO")


# ------------------------------------------------------------
# 4. Cargar bases
# ------------------------------------------------------------

saic = pd.read_csv(
    saic_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

denue = pd.read_csv(
    denue_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

print("\nBASES CARGADAS")
print("=" * 100)
print(f"SAIC filas: {saic.shape[0]:,} | columnas: {saic.shape[1]:,}")
print(f"DENUE filas: {denue.shape[0]:,} | columnas: {denue.shape[1]:,}")


# ------------------------------------------------------------
# 5. Función para homologar llaves
# ------------------------------------------------------------

def standardize_municipal_keys(df):
    """
    Homologa year, entidad_id, municipio_id y geo_key.
    Evita pérdida de ceros a la izquierda.
    """

    df = df.copy()

    df["year"] = pd.to_numeric(
        df["year"],
        errors="coerce"
    ).astype("Int64")

    df["entidad_id"] = (
        df["entidad_id"]
        .astype(str)
        .str.strip()
        .str.zfill(2)
    )

    df["municipio_id"] = (
        df["municipio_id"]
        .astype(str)
        .str.strip()
        .str.zfill(3)
    )

    df["geo_key"] = df["entidad_id"] + df["municipio_id"]

    return df


saic = standardize_municipal_keys(saic)
denue = standardize_municipal_keys(denue)


# ------------------------------------------------------------
# 6. Filtrar a años compatibles con DENUE
# ------------------------------------------------------------

years_to_keep = [2018, 2023]

saic_filtered = saic[
    saic["year"].isin(years_to_keep)
].copy()

denue_filtered = denue[
    denue["year"].isin(years_to_keep)
].copy()

print("\nAÑOS DISPONIBLES DESPUÉS DEL FILTRO")
print("=" * 100)

print("\nSAIC:")
display(
    saic_filtered["year"]
    .value_counts()
    .sort_index()
    .reset_index()
    .rename(columns={"index": "year", "year": "n_rows"})
)

print("\nDENUE:")
display(
    denue_filtered["year"]
    .value_counts()
    .sort_index()
    .reset_index()
    .rename(columns={"index": "year", "year": "n_rows"})
)


# ------------------------------------------------------------
# 7. Revisar columnas disponibles
# ------------------------------------------------------------

print("\nCOLUMNAS SAIC")
print("=" * 100)
for col in saic_filtered.columns:
    print(col)

print("\nCOLUMNAS DENUE")
print("=" * 100)
for col in denue_filtered.columns:
    print(col)


# ------------------------------------------------------------
# 8. Seleccionar columnas DENUE relevantes
# ------------------------------------------------------------

denue_columns_to_keep = [
    # Llaves
    "year",
    "geo_key",

    # Geografía DENUE de respaldo
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",

    # Totales
    "denue_total_establishments",
    "denue_total_scian_classes",

    # Manufactura DENUE como complemento
    "denue_manufacturing_establishments",

    # Oferta B2B por bloque
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_establishments",
    "denue_b2b_support_scian_classes",

    # Tamaño total
    "denue_micro_establishments",
    "denue_small_establishments",
    "denue_medium_establishments",
    "denue_large_establishments",
    "denue_medium_large_establishments",

    # Shares total
    "share_micro_establishments",
    "share_small_establishments",
    "share_medium_establishments",
    "share_large_establishments",
    "share_medium_large_establishments",

    # Tamaño B2B
    "denue_b2b_micro_establishments",
    "denue_b2b_small_establishments",
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",

    # Shares B2B
    "share_b2b_micro_establishments",
    "share_b2b_small_establishments",
    "share_b2b_medium_establishments",
    "share_b2b_large_establishments",
    "share_b2b_medium_large_establishments"
]

denue_columns_existing = [
    col for col in denue_columns_to_keep
    if col in denue_filtered.columns
]

denue_for_merge = denue_filtered[denue_columns_existing].copy()

print("\nCOLUMNAS DENUE QUE ENTRARÁN AL CRUCE")
print("=" * 100)
for col in denue_for_merge.columns:
    print(col)


# ------------------------------------------------------------
# 9. Revisar duplicados por llave
# ------------------------------------------------------------

saic_duplicates = (
    saic_filtered
    .duplicated(subset=["year", "geo_key"])
    .sum()
)

denue_duplicates = (
    denue_for_merge
    .duplicated(subset=["year", "geo_key"])
    .sum()
)

print("\nDUPLICADOS POR LLAVE year + geo_key")
print("=" * 100)
print(f"SAIC duplicados: {saic_duplicates}")
print(f"DENUE duplicados: {denue_duplicates}")

if saic_duplicates > 0:
    print("\nMuestra de duplicados SAIC:")
    display(
        saic_filtered[
            saic_filtered.duplicated(
                subset=["year", "geo_key"],
                keep=False
            )
        ]
        .sort_values(["year", "geo_key"])
        .head(30)
    )

if denue_duplicates > 0:
    print("\nMuestra de duplicados DENUE:")
    display(
        denue_for_merge[
            denue_for_merge.duplicated(
                subset=["year", "geo_key"],
                keep=False
            )
        ]
        .sort_values(["year", "geo_key"])
        .head(30)
    )


# ------------------------------------------------------------
# 10. Hacer cruce
# ------------------------------------------------------------

industrial_b2b_raw = saic_filtered.merge(
    denue_for_merge,
    on=["year", "geo_key"],
    how="left",
    suffixes=("", "_denue"),
    indicator=True
)

print("\nCRUCE REALIZADO")
print("=" * 100)
print(f"Filas: {industrial_b2b_raw.shape[0]:,}")
print(f"Columnas: {industrial_b2b_raw.shape[1]:,}")

print("\nEstado del merge:")
display(
    industrial_b2b_raw["_merge"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "merge_status", "_merge": "n_rows"})
)


# ------------------------------------------------------------
# 11. Validación del cruce por año
# ------------------------------------------------------------

merge_validation = (
    industrial_b2b_raw
    .groupby(["year", "_merge"], dropna=False)
    .size()
    .reset_index(name="n_rows")
)

merge_validation["share"] = (
    merge_validation["n_rows"] /
    merge_validation.groupby("year")["n_rows"].transform("sum")
)

print("\nVALIDACIÓN DEL MERGE POR AÑO")
print("=" * 100)
display(merge_validation)


# ------------------------------------------------------------
# 12. Crear bandera de match DENUE
# ------------------------------------------------------------

industrial_b2b_raw["has_denue_match"] = (
    industrial_b2b_raw["_merge"] == "both"
).astype(int)


# ------------------------------------------------------------
# 13. Rellenar variables DENUE numéricas faltantes con cero
# ------------------------------------------------------------
# Importante:
# Esto no modifica variables SAIC.
# Solo variables DENUE numéricas de conteos/shares.
# Se conserva has_denue_match para auditoría.

denue_id_cols = [
    "year",
    "geo_key",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name"
]

denue_numeric_cols = [
    col for col in denue_for_merge.columns
    if col not in denue_id_cols
]

for col in denue_numeric_cols:
    if col in industrial_b2b_raw.columns:
        industrial_b2b_raw[col] = pd.to_numeric(
            industrial_b2b_raw[col],
            errors="coerce"
        ).fillna(0)


# ------------------------------------------------------------
# 14. Crear banderas simples de disponibilidad B2B
# ------------------------------------------------------------

industrial_b2b_raw["has_b2b_support"] = (
    industrial_b2b_raw["denue_b2b_support_establishments"] > 0
).astype(int)

industrial_b2b_raw["has_b2b_medium_large"] = (
    industrial_b2b_raw["denue_b2b_medium_large_establishments"] > 0
).astype(int)

industrial_b2b_raw["has_b2b_large"] = (
    industrial_b2b_raw["denue_b2b_large_establishments"] > 0
).astype(int)


# ------------------------------------------------------------
# 15. Resumen del panel integrado
# ------------------------------------------------------------

summary_by_year = (
    industrial_b2b_raw
    .groupby("year")
    .agg(
        n_municipalities=("geo_key", "nunique"),
        municipalities_with_denue_match=("has_denue_match", "sum"),
        municipalities_with_b2b=("has_b2b_support", "sum"),
        municipalities_with_b2b_medium_large=("has_b2b_medium_large", "sum"),
        municipalities_with_b2b_large=("has_b2b_large", "sum"),
        total_b2b_support=("denue_b2b_support_establishments", "sum"),
        total_b2b_medium_large=("denue_b2b_medium_large_establishments", "sum"),
        total_b2b_large=("denue_b2b_large_establishments", "sum")
    )
    .reset_index()
)

print("\nRESUMEN DEL PANEL INTEGRADO POR AÑO")
print("=" * 100)
display(summary_by_year)


# ------------------------------------------------------------
# 16. Revisar muestra final
# ------------------------------------------------------------

print("\nMUESTRA DEL PANEL INTEGRADO RAW")
print("=" * 100)
display(industrial_b2b_raw.head(20))


# ------------------------------------------------------------
# 17. Guardar outputs
# ------------------------------------------------------------

industrial_b2b_raw.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

industrial_b2b_raw.to_csv(
    sheets_output_path,
    index=False,
    encoding="utf-8-sig"
)

merge_validation.to_csv(
    validation_path,
    index=False,
    encoding="utf-8-sig"
)

summary_by_year.to_csv(
    tables_path / "industrial_b2b_raw_summary_by_year.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nARCHIVOS GUARDADOS")
print("=" * 100)
print("Panel integrado raw:")
print(output_path)

print("\nVersión para Google Sheets:")
print(sheets_output_path)

print("\nValidación del merge:")
print(validation_path)

print("\nResumen por año:")
print(tables_path / "industrial_b2b_raw_summary_by_year.csv")


# ------------------------------------------------------------
# 18. Mensaje final
# ------------------------------------------------------------

print("\nPROCESO TERMINADO")
print("=" * 100)
print("El panel SAIC + DENUE quedó integrado.")
print("Aún no se crearon ratios ni tipologías finales.")
print("Siguiente paso: revisar columnas SAIC reales y definir variables analíticas.")

Mounted at /content/drive
VERIFICACIÓN DE ARCHIVOS
SAIC / Censos:
/content/drive/MyDrive/Nearshoring_Project/data/processed/municipality_industrial_capacity_panel.csv
OK

DENUE:
/content/drive/MyDrive/Nearshoring_Project/data/processed/municipality_denue_support_summary.csv
OK

BASES CARGADAS
SAIC filas: 12,038 | columnas: 38
DENUE filas: 4,905 | columnas: 34

AÑOS DISPONIBLES DESPUÉS DEL FILTRO

SAIC:


,n_rows,count
0,2018,2447
1,2023,2451



DENUE:


,n_rows,count
0,2018,2442
1,2023,2463



COLUMNAS SAIC
year
entidad_id
entidad_name
municipio_id
municipio_name
geo_key
industrial_capacity_score
industrial_capacity_percentile
rank_industrial_capacity_within_year
industrial_capacity_group
is_top_5_percent_within_year
is_top_10_percent_within_year
is_top_25_percent_within_year
total_manufacturing_establishments
total_manufacturing_employment
total_manufacturing_value_added
total_manufacturing_income
total_manufacturing_expenses
total_manufacturing_investment
total_manufacturing_gross_fixed_capital_formation
avg_manufacturing_employment_per_establishment
manufacturing_value_added_per_worker
manufacturing_income_per_worker
number_of_manufacturing_scian_activities
score_employment_within_year
score_value_added_within_year
score_income_within_year
score_establishments_within_year
score_industrial_diversity_within_year
score_avg_size_within_year
score_value_added_per_worker_within_year
rank_employment_within_year
rank_value_added_within_year
rank_income_within_year
rank_establish

...


RESUMEN DEL PANEL INTEGRADO POR AÑO


,year,n_municipalities,municipalities_with_denue_match,municipalities_with_b2b,municipalities_with_b2b_medium_large,municipalities_with_b2b_large,total_b2b_support,total_b2b_medium_large,total_b2b_large
0,2018,2447,2432,2264,659,382,248323.0,13601.0,4959.0
1,2023,2451,2442,2241,731,392,228833.0,15670.0,5828.0



MUESTRA DEL PANEL INTEGRADO RAW


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,industrial_capacity_score,industrial_capacity_percentile,rank_industrial_capacity_within_year,industrial_capacity_group,...,share_b2b_micro_establishments,share_b2b_small_establishments,share_b2b_medium_establishments,share_b2b_large_establishments,share_b2b_medium_large_establishments,_merge,has_denue_match,has_b2b_support,has_b2b_medium_large,has_b2b_large
0,2018,15,México,106,Toluca,15106,0.992818,1.000000,1.0,Top industrial hub,...,0.818220,0.121048,0.031198,0.029534,0.060732,both,1,1,1,1
1,2018,24,San Luis Potosí,028,San Luis Potosí,24028,0.992049,0.999555,2.0,Top industrial hub,...,0.755075,0.174853,0.040930,0.029142,0.070072,both,1,1,1,1
2,2018,02,Baja California,004,Tijuana,02004,0.990813,0.999109,3.0,Top industrial hub,...,0.763509,0.180819,0.036959,0.018713,0.055673,both,1,1,1,1
3,2018,22,Querétaro,014,Querétaro,22014,0.989939,0.998664,4.0,Top industrial hub,...,0.703641,0.223257,0.044313,0.028789,0.073102,both,1,1,1,1
4,2018,05,Coahuila de Zaragoza,035,Torreón,05035,0.989542,0.998218,5.0,Top industrial hub,...,0.710701,0.203277,0.045571,0.040451,0.086022,both,1,1,1,1
5,2018,08,Chihuahua,037,Juárez,08037,0.989231,0.997773,6.0,Top industrial hub,...,0.703513,0.204215,0.060890,0.031382,0.092272,both,1,1,1,1
6,2018,05,Coahuila de Zaragoza,030,Saltillo,05030,0.989228,0.997327,7.0,Top industrial hub,...,0.745647,0.182214,0.043532,0.028607,0.072139,both,1,1,1,1
7,2018,01,Aguascalientes,001,Aguascalientes,01001,0.988477,0.996882,8.0,Top industrial hub,...,0.769912,0.182437,0.033696,0.013955,0.047651,both,1,1,1,1
8,2018,14,Jalisco,120,Zapopan,14120,0.988180,0.996437,9.0,Top industrial hub,...,0.725440,0.204912,0.046554,0.023094,0.069648,both,1,1,1,1
9,2018,19,Nuevo León,039,Monterrey,19039,0.987723,0.995991,10.0,Top industrial hub,...,0.633101,0.253536,0.067342,0.046021,0.113363,both,1,1,1,1



ARCHIVOS GUARDADOS
Panel integrado raw:
/content/drive/MyDrive/Nearshoring_Project/data/processed/municipality_industrial_b2b_opportunity_panel_raw.csv

Versión para Google Sheets:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/industrial_b2b_opportunity_panel_raw_for_google_sheets.csv

Validación del merge:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/industrial_b2b_merge_validation_raw.csv

Resumen por año:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/industrial_b2b_raw_summary_by_year.csv

PROCESO TERMINADO
El panel SAIC + DENUE quedó integrado.
Aún no se crearon ratios ni tipologías finales.
Siguiente paso: revisar columnas SAIC reales y definir variables analíticas.


In [2]:
# ============================================================
# SCRIPT 2: VARIABLES ANALÍTICAS SAIC + DENUE
# Proyecto: Nearshoring_Project
#
# Input:
# - data/processed/municipality_industrial_b2b_opportunity_panel_raw.csv
#
# Outputs:
# - data/processed/municipality_industrial_b2b_opportunity_panel.csv
# - outputs/tables/industrial_b2b_typology_summary.csv
# - outputs/tables/industrial_b2b_top_opportunity_municipalities.csv
# - outputs/tables/industrial_b2b_analytics_for_google_sheets.csv
#
# Unidad:
# year × municipality
# ============================================================


# ------------------------------------------------------------
# 0. Montar Google Drive
# ------------------------------------------------------------

from google.colab import drive
drive.mount('/content/drive')


# ------------------------------------------------------------
# 1. Librerías
# ------------------------------------------------------------

import pandas as pd
import numpy as np
from pathlib import Path


# ------------------------------------------------------------
# 2. Rutas
# ------------------------------------------------------------

project_path = Path("/content/drive/MyDrive/Nearshoring_Project")

processed_path = project_path / "data" / "processed"
tables_path = project_path / "outputs" / "tables"

tables_path.mkdir(parents=True, exist_ok=True)

input_path = processed_path / "municipality_industrial_b2b_opportunity_panel_raw.csv"

output_path = processed_path / "municipality_industrial_b2b_opportunity_panel.csv"
sheets_output_path = tables_path / "industrial_b2b_analytics_for_google_sheets.csv"

typology_summary_path = tables_path / "industrial_b2b_typology_summary.csv"
top_opportunity_path = tables_path / "industrial_b2b_top_opportunity_municipalities.csv"
column_inventory_path = tables_path / "industrial_b2b_column_inventory.csv"


# ------------------------------------------------------------
# 3. Cargar panel raw
# ------------------------------------------------------------

panel = pd.read_csv(
    input_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

panel["year"] = pd.to_numeric(panel["year"], errors="coerce").astype("Int64")

panel["entidad_id"] = (
    panel["entidad_id"]
    .astype(str)
    .str.strip()
    .str.zfill(2)
)

panel["municipio_id"] = (
    panel["municipio_id"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

panel["geo_key"] = panel["entidad_id"] + panel["municipio_id"]

print("PANEL RAW CARGADO")
print("=" * 100)
print(f"Filas: {panel.shape[0]:,}")
print(f"Columnas: {panel.shape[1]:,}")

display(panel.head())


# ------------------------------------------------------------
# 4. Inventario de columnas
# ------------------------------------------------------------

column_inventory = pd.DataFrame({
    "column": panel.columns,
    "dtype": [str(panel[col].dtype) for col in panel.columns],
    "missing_values": [panel[col].isna().sum() for col in panel.columns],
    "missing_share": [panel[col].isna().mean() for col in panel.columns]
})

display(column_inventory)

column_inventory.to_csv(
    column_inventory_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nInventario de columnas guardado en:")
print(column_inventory_path)


# ------------------------------------------------------------
# 5. Detectar variables SAIC clave
# ------------------------------------------------------------
# Este bloque intenta detectar automáticamente las columnas centrales
# de capacidad manufacturera. Si alguna no se detecta, más abajo
# puedes fijarla manualmente.

def find_first_existing_column(df, candidates):
    """
    Devuelve la primera columna existente de una lista de candidatas.
    """
    for col in candidates:
        if col in df.columns:
            return col
    return None


industrial_capacity_percentile_col = find_first_existing_column(
    panel,
    [
        "industrial_capacity_percentile",
        "manufacturing_capacity_percentile",
        "capacity_percentile"
    ]
)

industrial_capacity_score_col = find_first_existing_column(
    panel,
    [
        "industrial_capacity_score",
        "manufacturing_capacity_score",
        "capacity_score"
    ]
)

manufacturing_establishments_col = find_first_existing_column(
    panel,
    [
        "total_manufacturing_establishments",
        "manufacturing_establishments",
        "establishments",
        "unidades_economicas",
        "UE_Unidades_economicas",
        "UE Unidades económicas"
    ]
)

manufacturing_employment_col = find_first_existing_column(
    panel,
    [
        "total_manufacturing_employment",
        "manufacturing_employment",
        "employment",
        "personal_ocupado_total",
        "H001A_Personal_ocupado_total",
        "H001A Personal ocupado total"
    ]
)

manufacturing_value_added_col = find_first_existing_column(
    panel,
    [
        "total_manufacturing_value_added",
        "manufacturing_value_added",
        "value_added",
        "valor_agregado_censal_bruto",
        "A131A_Valor_agregado_censal_bruto",
        "A131A Valor agregado censal bruto"
    ]
)

manufacturing_income_col = find_first_existing_column(
    panel,
    [
        "total_manufacturing_income",
        "manufacturing_income",
        "income",
        "ingresos",
        "A800A_Total_de_ingresos",
        "A800A Total de ingresos"
    ]
)

manufacturing_investment_col = find_first_existing_column(
    panel,
    [
        "total_manufacturing_investment",
        "manufacturing_investment",
        "investment",
        "inversion_total",
        "A211A_Inversion_total",
        "A211A Inversión total"
    ]
)

print("\nVARIABLES SAIC DETECTADAS")
print("=" * 100)
print(f"industrial_capacity_percentile_col: {industrial_capacity_percentile_col}")
print(f"industrial_capacity_score_col:      {industrial_capacity_score_col}")
print(f"manufacturing_establishments_col:   {manufacturing_establishments_col}")
print(f"manufacturing_employment_col:       {manufacturing_employment_col}")
print(f"manufacturing_value_added_col:      {manufacturing_value_added_col}")
print(f"manufacturing_income_col:           {manufacturing_income_col}")
print(f"manufacturing_investment_col:       {manufacturing_investment_col}")


# ------------------------------------------------------------
# 6. Opción manual si alguna columna no fue detectada
# ------------------------------------------------------------
# Si el print anterior muestra None para alguna variable importante,
# revisa el inventario de columnas y escribe manualmente el nombre exacto.
#
# Ejemplo:
# manufacturing_establishments_col = "nombre_real_de_la_columna"
# manufacturing_employment_col = "nombre_real_de_la_columna"
# industrial_capacity_percentile_col = "nombre_real_de_la_columna"

# Descomenta y ajusta si hace falta:
# industrial_capacity_percentile_col = "industrial_capacity_percentile"
# manufacturing_establishments_col = "total_manufacturing_establishments"
# manufacturing_employment_col = "total_manufacturing_employment"
# manufacturing_value_added_col = "total_manufacturing_value_added"


# ------------------------------------------------------------
# 7. Convertir variables relevantes a numéricas
# ------------------------------------------------------------

candidate_numeric_cols = [
    industrial_capacity_percentile_col,
    industrial_capacity_score_col,
    manufacturing_establishments_col,
    manufacturing_employment_col,
    manufacturing_value_added_col,
    manufacturing_income_col,
    manufacturing_investment_col,

    # DENUE
    "denue_total_establishments",
    "denue_manufacturing_establishments",
    "denue_b2b_support_establishments",
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_scian_classes",
    "denue_b2b_micro_establishments",
    "denue_b2b_small_establishments",
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",
    "share_b2b_micro_establishments",
    "share_b2b_small_establishments",
    "share_b2b_medium_establishments",
    "share_b2b_large_establishments",
    "share_b2b_medium_large_establishments"
]

candidate_numeric_cols = [
    col for col in candidate_numeric_cols
    if col is not None and col in panel.columns
]

for col in candidate_numeric_cols:
    panel[col] = pd.to_numeric(panel[col], errors="coerce")


# ------------------------------------------------------------
# 8. Crear indicadores básicos de disponibilidad DENUE
# ------------------------------------------------------------

panel["has_b2b_support"] = (
    panel["denue_b2b_support_establishments"] > 0
).astype(int)

panel["has_b2b_medium_large"] = (
    panel["denue_b2b_medium_large_establishments"] > 0
).astype(int)

panel["has_b2b_large"] = (
    panel["denue_b2b_large_establishments"] > 0
).astype(int)

panel["b2b_only_micro_small"] = (
    (panel["denue_b2b_support_establishments"] > 0) &
    (panel["denue_b2b_medium_large_establishments"] == 0)
).astype(int)


# ------------------------------------------------------------
# 9. Crear percentiles DENUE por año
# ------------------------------------------------------------
# Estos percentiles nos permiten comparar municipios dentro de cada año.

denue_percentile_vars = [
    "denue_b2b_support_establishments",
    "denue_b2b_medium_large_establishments",
    "denue_b2b_large_establishments",
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_scian_classes",
    "share_b2b_medium_large_establishments"
]

for var in denue_percentile_vars:
    if var in panel.columns:
        panel[f"{var}_percentile"] = (
            panel
            .groupby("year")[var]
            .rank(pct=True, method="average")
        )


# Nombres cortos útiles
panel["b2b_support_percentile"] = panel.get(
    "denue_b2b_support_establishments_percentile",
    np.nan
)

panel["b2b_medium_large_percentile"] = panel.get(
    "denue_b2b_medium_large_establishments_percentile",
    np.nan
)

panel["b2b_large_percentile"] = panel.get(
    "denue_b2b_large_establishments_percentile",
    np.nan
)


# ------------------------------------------------------------
# 10. Crear ratios SAIC-DENUE
# ------------------------------------------------------------
# Estos ratios sirven para aproximar oferta B2B relativa a la escala
# manufacturera del municipio.

def safe_ratio(numerator, denominator, multiplier=1):
    """
    Calcula numerator / denominator evitando división entre cero.
    """
    return (numerator / denominator.replace(0, np.nan)) * multiplier


# Ratios por establecimientos manufactureros SAIC
if manufacturing_establishments_col is not None:

    panel["b2b_support_per_100_manufacturing_establishments"] = safe_ratio(
        panel["denue_b2b_support_establishments"],
        panel[manufacturing_establishments_col],
        multiplier=100
    )

    panel["b2b_medium_large_per_100_manufacturing_establishments"] = safe_ratio(
        panel["denue_b2b_medium_large_establishments"],
        panel[manufacturing_establishments_col],
        multiplier=100
    )

    panel["b2b_large_per_100_manufacturing_establishments"] = safe_ratio(
        panel["denue_b2b_large_establishments"],
        panel[manufacturing_establishments_col],
        multiplier=100
    )

else:
    panel["b2b_support_per_100_manufacturing_establishments"] = np.nan
    panel["b2b_medium_large_per_100_manufacturing_establishments"] = np.nan
    panel["b2b_large_per_100_manufacturing_establishments"] = np.nan


# Ratios por empleo manufacturero SAIC
if manufacturing_employment_col is not None:

    panel["b2b_support_per_1000_manufacturing_workers"] = safe_ratio(
        panel["denue_b2b_support_establishments"],
        panel[manufacturing_employment_col],
        multiplier=1000
    )

    panel["b2b_medium_large_per_1000_manufacturing_workers"] = safe_ratio(
        panel["denue_b2b_medium_large_establishments"],
        panel[manufacturing_employment_col],
        multiplier=1000
    )

    panel["b2b_large_per_1000_manufacturing_workers"] = safe_ratio(
        panel["denue_b2b_large_establishments"],
        panel[manufacturing_employment_col],
        multiplier=1000
    )

else:
    panel["b2b_support_per_1000_manufacturing_workers"] = np.nan
    panel["b2b_medium_large_per_1000_manufacturing_workers"] = np.nan
    panel["b2b_large_per_1000_manufacturing_workers"] = np.nan


# Ratios por valor agregado manufacturero SAIC
# Nota: aquí multiplicamos por 1,000,000 si el valor agregado está en unidades monetarias grandes.
# Si tu variable está en miles de pesos, después podemos ajustar la escala.
if manufacturing_value_added_col is not None:

    panel["b2b_support_per_value_added"] = safe_ratio(
        panel["denue_b2b_support_establishments"],
        panel[manufacturing_value_added_col],
        multiplier=1
    )

    panel["b2b_medium_large_per_value_added"] = safe_ratio(
        panel["denue_b2b_medium_large_establishments"],
        panel[manufacturing_value_added_col],
        multiplier=1
    )

else:
    panel["b2b_support_per_value_added"] = np.nan
    panel["b2b_medium_large_per_value_added"] = np.nan


# Limpiar infinitos
ratio_cols = [
    "b2b_support_per_100_manufacturing_establishments",
    "b2b_medium_large_per_100_manufacturing_establishments",
    "b2b_large_per_100_manufacturing_establishments",
    "b2b_support_per_1000_manufacturing_workers",
    "b2b_medium_large_per_1000_manufacturing_workers",
    "b2b_large_per_1000_manufacturing_workers",
    "b2b_support_per_value_added",
    "b2b_medium_large_per_value_added"
]

for col in ratio_cols:
    panel[col] = panel[col].replace([np.inf, -np.inf], np.nan)


# ------------------------------------------------------------
# 11. Crear percentiles de ratios por año
# ------------------------------------------------------------

for col in ratio_cols:
    if col in panel.columns:
        panel[f"{col}_percentile"] = (
            panel
            .groupby("year")[col]
            .rank(pct=True, method="average")
        )


# ------------------------------------------------------------
# 12. Crear banderas high/low para tipología preliminar
# ------------------------------------------------------------
# Umbral inicial: percentil >= 0.75.
# Esta tipología es exploratoria y puede cambiar.

if industrial_capacity_percentile_col is not None:

    panel["high_industrial_capacity"] = (
        panel[industrial_capacity_percentile_col] >= 0.75
    ).astype(int)

else:
    panel["high_industrial_capacity"] = np.nan


panel["high_b2b_support"] = (
    panel["b2b_support_percentile"] >= 0.75
).astype(int)

panel["high_b2b_medium_large"] = (
    panel["b2b_medium_large_percentile"] >= 0.75
).astype(int)

panel["high_b2b_large"] = (
    panel["b2b_large_percentile"] >= 0.75
).astype(int)


# ------------------------------------------------------------
# 13. Tipología principal: capacidad industrial × B2B support
# ------------------------------------------------------------

def classify_broad_typology(row):
    """
    Tipología exploratoria usando:
    - high_industrial_capacity
    - high_b2b_support
    """

    if pd.isna(row["high_industrial_capacity"]):
        return "Not classified"

    if row["high_industrial_capacity"] == 1 and row["high_b2b_support"] == 1:
        return "Industrial ecosystem hub"

    elif row["high_industrial_capacity"] == 1 and row["high_b2b_support"] == 0:
        return "Potential underserved industrial market"

    elif row["high_industrial_capacity"] == 0 and row["high_b2b_support"] == 1:
        return "Service support hub"

    else:
        return "Lower initial priority"


panel["industrial_b2b_typology_broad"] = panel.apply(
    classify_broad_typology,
    axis=1
)


# ------------------------------------------------------------
# 14. Tipología alternativa: capacidad industrial × B2B medium/large
# ------------------------------------------------------------
# Esta versión es más exigente porque usa servicios con escala.

def classify_scaled_typology(row):
    """
    Tipología exploratoria usando:
    - high_industrial_capacity
    - high_b2b_medium_large
    """

    if pd.isna(row["high_industrial_capacity"]):
        return "Not classified"

    if row["high_industrial_capacity"] == 1 and row["high_b2b_medium_large"] == 1:
        return "Scaled industrial ecosystem hub"

    elif row["high_industrial_capacity"] == 1 and row["high_b2b_medium_large"] == 0:
        return "Potential scale gap in industrial services"

    elif row["high_industrial_capacity"] == 0 and row["high_b2b_medium_large"] == 1:
        return "Scaled service support hub"

    else:
        return "Lower initial priority"


panel["industrial_b2b_typology_scaled"] = panel.apply(
    classify_scaled_typology,
    axis=1
)


# ------------------------------------------------------------
# 15. Crear score exploratorio de oferta B2B
# ------------------------------------------------------------
# Score simple basado en percentiles de cantidad, diversidad y escala.
# No reemplaza análisis; sirve para ordenar municipios.

score_components = []

if "denue_b2b_support_establishments_percentile" in panel.columns:
    score_components.append("denue_b2b_support_establishments_percentile")

if "denue_b2b_support_scian_classes_percentile" in panel.columns:
    score_components.append("denue_b2b_support_scian_classes_percentile")

if "denue_b2b_medium_large_establishments_percentile" in panel.columns:
    score_components.append("denue_b2b_medium_large_establishments_percentile")

if "share_b2b_medium_large_establishments_percentile" in panel.columns:
    score_components.append("share_b2b_medium_large_establishments_percentile")

if len(score_components) > 0:
    panel["b2b_support_supply_score"] = panel[score_components].mean(axis=1)
    panel["b2b_support_supply_percentile"] = (
        panel
        .groupby("year")["b2b_support_supply_score"]
        .rank(pct=True, method="average")
    )
else:
    panel["b2b_support_supply_score"] = np.nan
    panel["b2b_support_supply_percentile"] = np.nan


# ------------------------------------------------------------
# 16. Tablas resumen de tipologías
# ------------------------------------------------------------

typology_summary = (
    panel
    .groupby(["year", "industrial_b2b_typology_broad"], dropna=False)
    .agg(
        n_municipalities=("geo_key", "nunique"),
        avg_b2b_support=("denue_b2b_support_establishments", "mean"),
        avg_b2b_medium_large=("denue_b2b_medium_large_establishments", "mean"),
        avg_share_b2b_medium_large=("share_b2b_medium_large_establishments", "mean")
    )
    .reset_index()
)

typology_summary["share_municipalities"] = (
    typology_summary["n_municipalities"] /
    typology_summary.groupby("year")["n_municipalities"].transform("sum")
)

print("\nRESUMEN DE TIPOLOGÍA BROAD")
print("=" * 100)
display(typology_summary)


scaled_typology_summary = (
    panel
    .groupby(["year", "industrial_b2b_typology_scaled"], dropna=False)
    .agg(
        n_municipalities=("geo_key", "nunique"),
        avg_b2b_support=("denue_b2b_support_establishments", "mean"),
        avg_b2b_medium_large=("denue_b2b_medium_large_establishments", "mean"),
        avg_share_b2b_medium_large=("share_b2b_medium_large_establishments", "mean")
    )
    .reset_index()
)

scaled_typology_summary["share_municipalities"] = (
    scaled_typology_summary["n_municipalities"] /
    scaled_typology_summary.groupby("year")["n_municipalities"].transform("sum")
)

print("\nRESUMEN DE TIPOLOGÍA SCALED")
print("=" * 100)
display(scaled_typology_summary)


# ------------------------------------------------------------
# 17. Top municipios de oportunidad preliminar
# ------------------------------------------------------------
# Enfoque:
# Municipios con alta capacidad industrial pero menor oferta B2B escalada.
# Esta tabla sirve para exploración, no como ranking final.

if industrial_capacity_percentile_col is not None:

    top_opportunity = (
        panel[
            (panel["high_industrial_capacity"] == 1)
        ]
        .copy()
    )

    # Ordenamos por alta capacidad industrial y baja oferta escalada relativa
    sort_cols = [
        industrial_capacity_percentile_col,
        "b2b_medium_large_percentile",
        "denue_b2b_medium_large_establishments"
    ]

    top_opportunity = (
        top_opportunity
        .sort_values(
            by=sort_cols,
            ascending=[False, True, True]
        )
        .head(100)
    )

    top_columns = [
        "year",
        "entidad_id",
        "entidad_name",
        "municipio_id",
        "municipio_name",
        "geo_key",
        industrial_capacity_percentile_col,
        industrial_capacity_score_col,
        manufacturing_establishments_col,
        manufacturing_employment_col,
        manufacturing_value_added_col,
        "denue_b2b_support_establishments",
        "denue_b2b_medium_large_establishments",
        "share_b2b_medium_large_establishments",
        "b2b_support_percentile",
        "b2b_medium_large_percentile",
        "industrial_b2b_typology_broad",
        "industrial_b2b_typology_scaled"
    ]

    top_columns = [
        col for col in top_columns
        if col is not None and col in top_opportunity.columns
    ]

    top_opportunity = top_opportunity[top_columns].copy()

else:
    top_opportunity = pd.DataFrame()

print("\nTOP MUNICIPIOS DE OPORTUNIDAD PRELIMINAR")
print("=" * 100)
display(top_opportunity.head(30))


# ------------------------------------------------------------
# 18. Resumen por año
# ------------------------------------------------------------

summary_by_year = (
    panel
    .groupby("year")
    .agg(
        n_municipalities=("geo_key", "nunique"),
        municipalities_with_b2b=("has_b2b_support", "sum"),
        municipalities_with_b2b_medium_large=("has_b2b_medium_large", "sum"),
        municipalities_high_industrial_capacity=("high_industrial_capacity", "sum"),
        total_b2b_support=("denue_b2b_support_establishments", "sum"),
        total_b2b_medium_large=("denue_b2b_medium_large_establishments", "sum"),
        avg_share_b2b_medium_large=("share_b2b_medium_large_establishments", "mean")
    )
    .reset_index()
)

print("\nRESUMEN GENERAL POR AÑO")
print("=" * 100)
display(summary_by_year)


# ------------------------------------------------------------
# 19. Guardar outputs
# ------------------------------------------------------------

panel.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

panel.to_csv(
    sheets_output_path,
    index=False,
    encoding="utf-8-sig"
)

typology_summary.to_csv(
    typology_summary_path,
    index=False,
    encoding="utf-8-sig"
)

scaled_typology_summary.to_csv(
    tables_path / "industrial_b2b_scaled_typology_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

top_opportunity.to_csv(
    top_opportunity_path,
    index=False,
    encoding="utf-8-sig"
)

summary_by_year.to_csv(
    tables_path / "industrial_b2b_analytics_summary_by_year.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nARCHIVOS GUARDADOS")
print("=" * 100)
print("Panel analítico:")
print(output_path)

print("\nVersión para Google Sheets:")
print(sheets_output_path)

print("\nResumen tipología broad:")
print(typology_summary_path)

print("\nResumen tipología scaled:")
print(tables_path / "industrial_b2b_scaled_typology_summary.csv")

print("\nTop municipios preliminares:")
print(top_opportunity_path)

print("\nResumen por año:")
print(tables_path / "industrial_b2b_analytics_summary_by_year.csv")


# ------------------------------------------------------------
# 20. Mensaje final
# ------------------------------------------------------------

print("\nPROCESO TERMINADO")
print("=" * 100)
print("Panel analítico SAIC + DENUE creado.")
print("Revisa primero las variables SAIC detectadas antes de interpretar ratios o tipologías.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PANEL RAW CARGADO
Filas: 4,898
Columnas: 75


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,industrial_capacity_score,industrial_capacity_percentile,rank_industrial_capacity_within_year,industrial_capacity_group,...,share_b2b_micro_establishments,share_b2b_small_establishments,share_b2b_medium_establishments,share_b2b_large_establishments,share_b2b_medium_large_establishments,_merge,has_denue_match,has_b2b_support,has_b2b_medium_large,has_b2b_large
0,2018,15,México,106,Toluca,15106,0.992818,1.000000,1.0,Top industrial hub,...,0.818220,0.121048,0.031198,0.029534,0.060732,both,1,1,1,1
1,2018,24,San Luis Potosí,028,San Luis Potosí,24028,0.992049,0.999555,2.0,Top industrial hub,...,0.755075,0.174853,0.040930,0.029142,0.070072,both,1,1,1,1
2,2018,02,Baja California,004,Tijuana,02004,0.990813,0.999109,3.0,Top industrial hub,...,0.763509,0.180819,0.036959,0.018713,0.055673,both,1,1,1,1
3,2018,22,Querétaro,014,Querétaro,22014,0.989939,0.998664,4.0,Top industrial hub,...,0.703641,0.223257,0.044313,0.028789,0.073102,both,1,1,1,1
4,2018,05,Coahuila de Zaragoza,035,Torreón,05035,0.989542,0.998218,5.0,Top industrial hub,...,0.710701,0.203277,0.045571,0.040451,0.086022,both,1,1,1,1


,column,dtype,missing_values,missing_share
0,year,Int64,0,0.0
1,entidad_id,object,0,0.0
2,entidad_name,object,0,0.0
3,municipio_id,object,0,0.0
4,municipio_name,object,0,0.0
...,...,...,...,...
70,_merge,object,0,0.0
71,has_denue_match,int64,0,0.0
72,has_b2b_support,int64,0,0.0
73,has_b2b_medium_large,int64,0,0.0



Inventario de columnas guardado en:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/industrial_b2b_column_inventory.csv

VARIABLES SAIC DETECTADAS
industrial_capacity_percentile_col: industrial_capacity_percentile
industrial_capacity_score_col:      industrial_capacity_score
manufacturing_establishments_col:   total_manufacturing_establishments
manufacturing_employment_col:       total_manufacturing_employment
manufacturing_value_added_col:      total_manufacturing_value_added
manufacturing_income_col:           total_manufacturing_income
manufacturing_investment_col:       total_manufacturing_investment

RESUMEN DE TIPOLOGÍA BROAD


,year,industrial_b2b_typology_broad,n_municipalities,avg_b2b_support,avg_b2b_medium_large,avg_share_b2b_medium_large,share_municipalities
0,2018,Industrial ecosystem hub,455,475.481319,28.331868,0.036794,0.185942
1,2018,Lower initial priority,1724,8.798144,0.100928,0.008643,0.704536
2,2018,Potential underserved industrial market,107,25.626168,0.719626,0.028901,0.043727
3,2018,Service support hub,161,87.385093,2.850932,0.022247,0.065795
4,2023,Industrial ecosystem hub,456,440.883772,33.089912,0.047131,0.186047
5,2023,Lower initial priority,1724,8.342227,0.142691,0.013514,0.703386
6,2023,Potential underserved industrial market,116,24.456897,0.827586,0.035864,0.047328
7,2023,Service support hub,155,68.200000,1.541935,0.023199,0.063239


...


TOP MUNICIPIOS DE OPORTUNIDAD PRELIMINAR


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,industrial_capacity_percentile,industrial_capacity_score,total_manufacturing_establishments,total_manufacturing_employment,total_manufacturing_value_added,denue_b2b_support_establishments,denue_b2b_medium_large_establishments,share_b2b_medium_large_establishments,b2b_support_percentile,b2b_medium_large_percentile,industrial_b2b_typology_broad,industrial_b2b_typology_scaled
2447,2023,05,Coahuila de Zaragoza,030,Saltillo,05030,1.000000,0.992587,2758.0,74640.0,226838.388,1542.0,167.0,0.108301,0.986536,0.989596,Industrial ecosystem hub,Scaled industrial ecosystem hub
0,2018,15,México,106,Toluca,15106,1.000000,0.992818,4351.0,83598.0,97605.998,2404.0,146.0,0.060732,0.993461,0.991009,Industrial ecosystem hub,Scaled industrial ecosystem hub
2448,2023,24,San Luis Potosí,028,San Luis Potosí,24028,0.999562,0.991693,3545.0,122990.0,112107.049,2779.0,284.0,0.102195,0.995920,0.996328,Industrial ecosystem hub,Scaled industrial ecosystem hub
1,2018,24,San Luis Potosí,028,San Luis Potosí,24028,0.999555,0.992049,3824.0,126640.0,77223.073,3054.0,214.0,0.070072,0.995913,0.995913,Industrial ecosystem hub,Scaled industrial ecosystem hub
2449,2023,22,Querétaro,014,Querétaro,22014,0.999125,0.991194,3544.0,112895.0,111460.482,3191.0,349.0,0.109370,0.996736,0.997960,Industrial ecosystem hub,Scaled industrial ecosystem hub
2,2018,02,Baja California,004,Tijuana,02004,0.999109,0.990813,3670.0,270055.0,85526.588,4275.0,238.0,0.055673,0.997957,0.996731,Industrial ecosystem hub,Scaled industrial ecosystem hub
2450,2023,01,Aguascalientes,001,Aguascalientes,01001,0.998687,0.990765,4166.0,75556.0,128718.721,2673.0,167.0,0.062477,0.995104,0.989596,Industrial ecosystem hub,Scaled industrial ecosystem hub
3,2018,22,Querétaro,014,Querétaro,22014,0.998664,0.989939,3363.0,103369.0,60661.916,3543.0,259.0,0.073102,0.996731,0.997548,Industrial ecosystem hub,Scaled industrial ecosystem hub
2451,2023,15,México,106,Toluca,15106,0.998249,0.990511,5132.0,97042.0,99527.258,2458.0,179.0,0.072823,0.994288,0.992656,Industrial ecosystem hub,Scaled industrial ecosystem hub
4,2018,05,Coahuila de Zaragoza,035,Torreón,05035,0.998218,0.989542,2046.0,70589.0,72451.993,1953.0,168.0,0.086022,0.989783,0.993053,Industrial ecosystem hub,Scaled industrial ecosystem hub



RESUMEN GENERAL POR AÑO


,year,n_municipalities,municipalities_with_b2b,municipalities_with_b2b_medium_large,municipalities_high_industrial_capacity,total_b2b_support,total_b2b_medium_large,avg_share_b2b_medium_large
0,2018,2447,2264,659,562,248323.0,13601.0,0.015658
1,2023,2451,2241,731,572,228833.0,15670.0,0.021438



ARCHIVOS GUARDADOS
Panel analítico:
/content/drive/MyDrive/Nearshoring_Project/data/processed/municipality_industrial_b2b_opportunity_panel.csv

Versión para Google Sheets:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/industrial_b2b_analytics_for_google_sheets.csv

Resumen tipología broad:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/industrial_b2b_typology_summary.csv

Resumen tipología scaled:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/industrial_b2b_scaled_typology_summary.csv

Top municipios preliminares:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/industrial_b2b_top_opportunity_municipalities.csv

Resumen por año:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/industrial_b2b_analytics_summary_by_year.csv

PROCESO TERMINADO
Panel analítico SAIC + DENUE creado.
Revisa primero las variables SAIC detectadas antes de interpretar ratios o tipologías.


In [3]:
# ============================================================
# SCRIPT 3: FILTRAR COLUMNAS DEL PANEL ANALÍTICO SAIC + DENUE
# Proyecto: Nearshoring_Project
#
# Input:
# data/processed/municipality_industrial_b2b_opportunity_panel.csv
#
# Outputs:
# data/processed/municipality_industrial_b2b_opportunity_panel_curated.csv
# outputs/tables/industrial_b2b_curated_for_google_sheets.csv
# outputs/tables/industrial_b2b_curated_column_dictionary.csv
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 1. Rutas
# ------------------------------------------------------------

project_path = Path("/content/drive/MyDrive/Nearshoring_Project")

processed_path = project_path / "data" / "processed"
tables_path = project_path / "outputs" / "tables"

tables_path.mkdir(parents=True, exist_ok=True)

input_path = processed_path / "municipality_industrial_b2b_opportunity_panel.csv"

output_path = processed_path / "municipality_industrial_b2b_opportunity_panel_curated.csv"

sheets_output_path = tables_path / "industrial_b2b_curated_for_google_sheets.csv"

dictionary_output_path = tables_path / "industrial_b2b_curated_column_dictionary.csv"


# ------------------------------------------------------------
# 2. Cargar panel analítico
# ------------------------------------------------------------

panel = pd.read_csv(
    input_path,
    dtype={
        "year": str,
        "entidad_id": str,
        "municipio_id": str,
        "geo_key": str
    },
    low_memory=False
)

panel["year"] = pd.to_numeric(panel["year"], errors="coerce").astype("Int64")
panel["entidad_id"] = panel["entidad_id"].astype(str).str.strip().str.zfill(2)
panel["municipio_id"] = panel["municipio_id"].astype(str).str.strip().str.zfill(3)
panel["geo_key"] = panel["entidad_id"] + panel["municipio_id"]

print("Panel cargado")
print("=" * 100)
print(f"Filas: {panel.shape[0]:,}")
print(f"Columnas originales: {panel.shape[1]:,}")


# ------------------------------------------------------------
# 3. Definir columnas a conservar
# ------------------------------------------------------------

columns_to_keep = [
    # --------------------------------------------------------
    # Identificación geográfica
    # --------------------------------------------------------
    "year",
    "entidad_id",
    "entidad_name",
    "municipio_id",
    "municipio_name",
    "geo_key",

    # --------------------------------------------------------
    # SAIC / capacidad manufacturera
    # --------------------------------------------------------
    "industrial_capacity_score",
    "industrial_capacity_percentile",
    "rank_industrial_capacity_within_year",
    "industrial_capacity_group",
    "total_manufacturing_establishments",
    "total_manufacturing_employment",
    "total_manufacturing_value_added",
    "total_manufacturing_income",
    "total_manufacturing_investment",
    "manufacturing_diversity_activities",

    # --------------------------------------------------------
    # DENUE / complemento manufacturero
    # --------------------------------------------------------
    "denue_manufacturing_establishments",

    # --------------------------------------------------------
    # DENUE / oferta B2B central
    # --------------------------------------------------------
    "denue_b2b_support_establishments",
    "denue_logistics_storage_establishments",
    "denue_professional_technical_establishments",
    "denue_business_support_establishments",
    "denue_b2b_support_scian_classes",

    # --------------------------------------------------------
    # DENUE / tamaño de establecimientos B2B
    # --------------------------------------------------------
    "denue_b2b_micro_establishments",
    "denue_b2b_small_establishments",
    "denue_b2b_medium_establishments",
    "denue_b2b_large_establishments",
    "denue_b2b_medium_large_establishments",

    # --------------------------------------------------------
    # DENUE / composición de tamaño B2B
    # --------------------------------------------------------
    "share_b2b_micro_establishments",
    "share_b2b_small_establishments",
    "share_b2b_medium_establishments",
    "share_b2b_large_establishments",
    "share_b2b_medium_large_establishments",

    # --------------------------------------------------------
    # Banderas interpretativas
    # --------------------------------------------------------
    "has_denue_match",
    "has_b2b_support",
    "has_b2b_medium_large",
    "has_b2b_large",
    "b2b_only_micro_small",

    # --------------------------------------------------------
    # Percentiles DENUE principales
    # --------------------------------------------------------
    "b2b_support_percentile",
    "b2b_medium_large_percentile",
    "b2b_large_percentile",
    "b2b_support_supply_score",
    "b2b_support_supply_percentile",

    # --------------------------------------------------------
    # Ratios SAIC-DENUE por establecimientos manufactureros
    # --------------------------------------------------------
    "b2b_support_per_100_manufacturing_establishments",
    "b2b_medium_large_per_100_manufacturing_establishments",
    "b2b_large_per_100_manufacturing_establishments",

    # --------------------------------------------------------
    # Ratios SAIC-DENUE por empleo manufacturero
    # --------------------------------------------------------
    "b2b_support_per_1000_manufacturing_workers",
    "b2b_medium_large_per_1000_manufacturing_workers",
    "b2b_large_per_1000_manufacturing_workers",

    # --------------------------------------------------------
    # Tipologías
    # --------------------------------------------------------
    "high_industrial_capacity",
    "high_b2b_support",
    "high_b2b_medium_large",
    "high_b2b_large",
    "industrial_b2b_typology_broad",
    "industrial_b2b_typology_scaled"
]


# Mantener solo columnas existentes, para evitar errores si alguna no existe
columns_existing = [col for col in columns_to_keep if col in panel.columns]

missing_requested_columns = [col for col in columns_to_keep if col not in panel.columns]

panel_curated = panel[columns_existing].copy()

print("\nFiltrado terminado")
print("=" * 100)
print(f"Columnas conservadas: {len(columns_existing):,}")
print(f"Columnas eliminadas: {panel.shape[1] - len(columns_existing):,}")

if len(missing_requested_columns) > 0:
    print("\nColumnas solicitadas que NO existen en el panel:")
    for col in missing_requested_columns:
        print("-", col)


# ------------------------------------------------------------
# 4. Diccionario de columnas curadas
# ------------------------------------------------------------

column_descriptions = {
    "year": "Año de referencia del panel.",
    "entidad_id": "Clave de entidad federativa.",
    "entidad_name": "Nombre de la entidad federativa.",
    "municipio_id": "Clave municipal.",
    "municipio_name": "Nombre del municipio.",
    "geo_key": "Clave municipal de cinco dígitos, compatible con CVEGEO.",

    "industrial_capacity_score": "Score de capacidad manufacturera construido con variables SAIC/Censos.",
    "industrial_capacity_percentile": "Percentil del score de capacidad manufacturera dentro de cada año.",
    "rank_industrial_capacity_within_year": "Ranking municipal de capacidad manufacturera dentro de cada año.",
    "industrial_capacity_group": "Grupo categórico de capacidad manufacturera.",
    "total_manufacturing_establishments": "Total de unidades económicas manufactureras según SAIC/Censos.",
    "total_manufacturing_employment": "Personal ocupado manufacturero total según SAIC/Censos.",
    "total_manufacturing_value_added": "Valor agregado manufacturero total según SAIC/Censos.",
    "total_manufacturing_income": "Ingresos manufactureros totales según SAIC/Censos.",
    "total_manufacturing_investment": "Inversión manufacturera total según SAIC/Censos.",
    "manufacturing_diversity_activities": "Número de actividades manufactureras presentes en el municipio.",

    "denue_manufacturing_establishments": "Establecimientos manufactureros DENUE; usado como complemento territorial, no como oferta B2B central.",

    "denue_b2b_support_establishments": "Total de establecimientos DENUE en servicios B2B de soporte industrial: 48-49, 54 y 56.",
    "denue_logistics_storage_establishments": "Establecimientos DENUE de transporte, correos y almacenamiento.",
    "denue_professional_technical_establishments": "Establecimientos DENUE de servicios profesionales, científicos y técnicos.",
    "denue_business_support_establishments": "Establecimientos DENUE de apoyo a negocios, manejo de residuos y remediación.",
    "denue_b2b_support_scian_classes": "Número de clases SCIAN B2B observadas en el municipio.",

    "denue_b2b_micro_establishments": "Servicios B2B DENUE con 0 a 5 personas ocupadas.",
    "denue_b2b_small_establishments": "Servicios B2B DENUE con 6 a 30 personas ocupadas.",
    "denue_b2b_medium_establishments": "Servicios B2B DENUE con 31 a 100 personas ocupadas.",
    "denue_b2b_large_establishments": "Servicios B2B DENUE con 101 o más personas ocupadas.",
    "denue_b2b_medium_large_establishments": "Servicios B2B DENUE medianos o grandes.",

    "share_b2b_micro_establishments": "Participación de servicios B2B micro sobre el total B2B municipal.",
    "share_b2b_small_establishments": "Participación de servicios B2B pequeños sobre el total B2B municipal.",
    "share_b2b_medium_establishments": "Participación de servicios B2B medianos sobre el total B2B municipal.",
    "share_b2b_large_establishments": "Participación de servicios B2B grandes sobre el total B2B municipal.",
    "share_b2b_medium_large_establishments": "Participación de servicios B2B medianos y grandes sobre el total B2B municipal.",

    "has_denue_match": "Indica si el municipio encontró match con DENUE en el cruce.",
    "has_b2b_support": "Indica si el municipio tiene al menos un establecimiento B2B DENUE.",
    "has_b2b_medium_large": "Indica si el municipio tiene al menos un establecimiento B2B mediano o grande.",
    "has_b2b_large": "Indica si el municipio tiene al menos un establecimiento B2B grande.",
    "b2b_only_micro_small": "Indica si el municipio tiene B2B, pero solo micro o pequeños.",

    "b2b_support_percentile": "Percentil municipal de establecimientos B2B dentro de cada año.",
    "b2b_medium_large_percentile": "Percentil municipal de establecimientos B2B medianos/grandes dentro de cada año.",
    "b2b_large_percentile": "Percentil municipal de establecimientos B2B grandes dentro de cada año.",
    "b2b_support_supply_score": "Score exploratorio de oferta B2B basado en cantidad, diversidad y escala.",
    "b2b_support_supply_percentile": "Percentil del score exploratorio de oferta B2B dentro de cada año.",

    "b2b_support_per_100_manufacturing_establishments": "Servicios B2B por cada 100 establecimientos manufactureros SAIC.",
    "b2b_medium_large_per_100_manufacturing_establishments": "Servicios B2B medianos/grandes por cada 100 establecimientos manufactureros SAIC.",
    "b2b_large_per_100_manufacturing_establishments": "Servicios B2B grandes por cada 100 establecimientos manufactureros SAIC.",
    "b2b_support_per_1000_manufacturing_workers": "Servicios B2B por cada 1,000 trabajadores manufactureros SAIC.",
    "b2b_medium_large_per_1000_manufacturing_workers": "Servicios B2B medianos/grandes por cada 1,000 trabajadores manufactureros SAIC.",
    "b2b_large_per_1000_manufacturing_workers": "Servicios B2B grandes por cada 1,000 trabajadores manufactureros SAIC.",

    "high_industrial_capacity": "Bandera de alta capacidad manufacturera; percentil industrial >= 0.75.",
    "high_b2b_support": "Bandera de alta oferta B2B; percentil B2B >= 0.75.",
    "high_b2b_medium_large": "Bandera de alta oferta B2B mediana/grande; percentil >= 0.75.",
    "high_b2b_large": "Bandera de alta oferta B2B grande; percentil >= 0.75.",
    "industrial_b2b_typology_broad": "Tipología exploratoria basada en capacidad manufacturera y volumen B2B.",
    "industrial_b2b_typology_scaled": "Tipología exploratoria basada en capacidad manufacturera y B2B mediano/grande."
}

column_dictionary = pd.DataFrame({
    "column": panel_curated.columns,
    "description": [
        column_descriptions.get(col, "")
        for col in panel_curated.columns
    ]
})


# ------------------------------------------------------------
# 5. Validaciones rápidas
# ------------------------------------------------------------

print("\nVista del panel curado")
print("=" * 100)
display(panel_curated.head(20))

print("\nResumen de columnas del panel curado")
print("=" * 100)
display(column_dictionary)

print("\nFilas por año")
display(
    panel_curated
    .groupby("year")
    .size()
    .reset_index(name="n_rows")
)


# ------------------------------------------------------------
# 6. Guardar outputs
# ------------------------------------------------------------

panel_curated.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

panel_curated.to_csv(
    sheets_output_path,
    index=False,
    encoding="utf-8-sig"
)

column_dictionary.to_csv(
    dictionary_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nARCHIVOS GUARDADOS")
print("=" * 100)
print("Panel curado:")
print(output_path)

print("\nVersión para Google Sheets:")
print(sheets_output_path)

print("\nDiccionario de columnas:")
print(dictionary_output_path)

print("\nPROCESO TERMINADO")
print("=" * 100)
print("Panel curado listo para análisis, mapas y reporte.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Panel cargado
Filas: 4,898
Columnas originales: 111

Filtrado terminado
Columnas conservadas: 53
Columnas eliminadas: 58

Columnas solicitadas que NO existen en el panel:
- manufacturing_diversity_activities

Vista del panel curado


,year,entidad_id,entidad_name,municipio_id,municipio_name,geo_key,industrial_capacity_score,industrial_capacity_percentile,rank_industrial_capacity_within_year,industrial_capacity_group,...,b2b_large_per_100_manufacturing_establishments,b2b_support_per_1000_manufacturing_workers,b2b_medium_large_per_1000_manufacturing_workers,b2b_large_per_1000_manufacturing_workers,high_industrial_capacity,high_b2b_support,high_b2b_medium_large,high_b2b_large,industrial_b2b_typology_broad,industrial_b2b_typology_scaled
0,2018,15,México,106,Toluca,15106,0.992818,1.000000,1.0,Top industrial hub,...,1.631809,28.756669,1.746453,0.849303,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
1,2018,24,San Luis Potosí,028,San Luis Potosí,24028,0.992049,0.999555,2.0,Top industrial hub,...,2.327406,24.115603,1.689829,0.702780,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
2,2018,02,Baja California,004,Tijuana,02004,0.990813,0.999109,3.0,Top industrial hub,...,2.179837,15.830109,0.881302,0.296236,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
3,2018,22,Querétaro,014,Querétaro,22014,0.989939,0.998664,4.0,Top industrial hub,...,3.033006,34.275266,2.505587,0.986756,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
4,2018,05,Coahuila de Zaragoza,035,Torreón,05035,0.989542,0.998218,5.0,Top industrial hub,...,3.861193,27.667200,2.379974,1.119155,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
5,2018,08,Chihuahua,037,Juárez,08037,0.989231,0.997773,6.0,Top industrial hub,...,2.821053,6.475093,0.597468,0.203200,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
6,2018,05,Coahuila de Zaragoza,030,Saltillo,05030,0.989228,0.997327,7.0,Top industrial hub,...,1.733233,22.351649,1.612432,0.639413,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
7,2018,01,Aguascalientes,001,Aguascalientes,01001,0.988477,0.996882,8.0,Top industrial hub,...,1.052361,41.310461,1.968504,0.576490,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
8,2018,14,Jalisco,120,Zapopan,14120,0.988180,0.996437,9.0,Top industrial hub,...,1.433447,22.410068,1.560819,0.517535,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub
9,2018,19,Nuevo León,039,Monterrey,19039,0.987723,0.995991,10.0,Top industrial hub,...,4.609854,62.625595,7.099418,2.882073,1,1,1,1,Industrial ecosystem hub,Scaled industrial ecosystem hub



Resumen de columnas del panel curado


,column,description
0,year,Año de referencia del panel.
1,entidad_id,Clave de entidad federativa.
2,entidad_name,Nombre de la entidad federativa.
3,municipio_id,Clave municipal.
4,municipio_name,Nombre del municipio.
5,geo_key,"Clave municipal de cinco dígitos, compatible c..."
6,industrial_capacity_score,Score de capacidad manufacturera construido co...
7,industrial_capacity_percentile,Percentil del score de capacidad manufacturera...
8,rank_industrial_capacity_within_year,Ranking municipal de capacidad manufacturera d...
9,industrial_capacity_group,Grupo categórico de capacidad manufacturera.



Filas por año


,year,n_rows
0,2018,2447
1,2023,2451



ARCHIVOS GUARDADOS
Panel curado:
/content/drive/MyDrive/Nearshoring_Project/data/processed/municipality_industrial_b2b_opportunity_panel_curated.csv

Versión para Google Sheets:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/industrial_b2b_curated_for_google_sheets.csv

Diccionario de columnas:
/content/drive/MyDrive/Nearshoring_Project/outputs/tables/industrial_b2b_curated_column_dictionary.csv

PROCESO TERMINADO
Panel curado listo para análisis, mapas y reporte.


In [4]:
# ============================================================
# BUSCAR ARCHIVOS RELACIONADOS CON EL PANEL INDUSTRIAL-B2B
# ============================================================

from pathlib import Path
import pandas as pd

drive_root = Path("/content/drive/MyDrive")

search_terms = [
    "municipality_industrial_b2b",
    "industrial_b2b",
    "opportunity_panel",
    "curated"
]

allowed_extensions = [".csv", ".xlsx", ".xls", ".parquet"]

matches = []

for path in drive_root.rglob("*"):
    if path.is_file() and path.suffix.lower() in allowed_extensions:
        name_lower = path.name.lower()
        if any(term.lower() in name_lower for term in search_terms):
            matches.append(path)

print(f"Coincidencias encontradas: {len(matches)}")
print("=" * 100)

for i, path in enumerate(matches):
    print(f"{i} -> {path}")

Coincidencias encontradas: 16
0 -> /content/drive/MyDrive/Nearshoring_Project/data/processed/municipality_industrial_b2b_opportunity_panel_raw.csv
1 -> /content/drive/MyDrive/Nearshoring_Project/data/processed/municipality_industrial_b2b_opportunity_panel.csv
2 -> /content/drive/MyDrive/Nearshoring_Project/data/processed/municipality_industrial_b2b_opportunity_panel_curated.csv
3 -> /content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/final_scores/01_municipality_industrial_b2b_opportunity_scores.csv
4 -> /content/drive/MyDrive/Nearshoring_Project/data/processed/modeling_base/final_scores/01_municipality_industrial_b2b_opportunity_scores.parquet
5 -> /content/drive/MyDrive/Nearshoring_Project/outputs/tables/industrial_b2b_opportunity_panel_raw_for_google_sheets.csv
6 -> /content/drive/MyDrive/Nearshoring_Project/outputs/tables/industrial_b2b_merge_validation_raw.csv
7 -> /content/drive/MyDrive/Nearshoring_Project/outputs/tables/industrial_b2b_raw_summary_by_year.csv


# ============================================================
# DIAGNÓSTICO DE ARCHIVOS DEL PROYECTO
# Nearshoring_Project
# ============================================================

from pathlib import Path
import pandas as pd
import os
from datetime import datetime

# ------------------------------------------------------------
# 1. Definir rutas candidatas
# ------------------------------------------------------------

drive_root = Path("/content/drive/MyDrive")

project_candidates = [
    drive_root / "Nearshoring_Project",
    drive_root / "nearshoring_project",
    drive_root / "Nearshoring Project",
    drive_root / "Nearshoring"
]

print("RUTAS CANDIDATAS DEL PROYECTO")
print("=" * 100)

for p in project_candidates:
    print(p, "->", "EXISTE" if p.exists() else "NO EXISTE")


# ------------------------------------------------------------
# 2. Buscar carpetas que contengan 'nearshoring'
# ------------------------------------------------------------

print("\nBUSCANDO CARPETAS CON 'nearshoring' EN MYDRIVE")
print("=" * 100)

nearshoring_dirs = [
    p for p in drive_root.rglob("*")
    if p.is_dir() and "nearshoring" in p.name.lower()
]

for i, p in enumerate(nearshoring_dirs):
    print(f"{i} -> {p}")

if len(nearshoring_dirs) == 0:
    print("No encontré carpetas con 'nearshoring'.")


# ------------------------------------------------------------
# 3. Usar la carpeta esperada si existe
# ------------------------------------------------------------

project_path = drive_root / "Nearshoring_Project"

if not project_path.exists():
    raise FileNotFoundError(
        "No encontré /content/drive/MyDrive/Nearshoring_Project. "
        "Revisa si el proyecto está en otra carpeta o cuenta de Drive."
    )


# ------------------------------------------------------------
# 4. Listar estructura inmediata del proyecto
# ------------------------------------------------------------

print("\nCONTENIDO INMEDIATO DE Nearshoring_Project")
print("=" * 100)

for item in project_path.iterdir():
    print(item)


# ------------------------------------------------------------
# 5. Revisar carpetas clave
# ------------------------------------------------------------

key_dirs = [
    project_path / "data",
    project_path / "data" / "processed",
    project_path / "outputs",
    project_path / "outputs" / "tables",
    project_path / "data" / "Geo"
]

print("\nCARPETAS CLAVE")
print("=" * 100)

for d in key_dirs:
    print(d, "->", "EXISTE" if d.exists() else "NO EXISTE")


# ------------------------------------------------------------
# 6. Listar archivos en data/processed y outputs/tables
# ------------------------------------------------------------

for d in [project_path / "data" / "processed", project_path / "outputs" / "tables"]:
    print(f"\nARCHIVOS EN: {d}")
    print("=" * 100)

    if d.exists():
        files = list(d.glob("*"))
        print(f"Total archivos/carpetas: {len(files)}")

        for f in files:
            print(f.name)
    else:
        print("No existe esta carpeta.")


# ------------------------------------------------------------
# 7. Buscar TODOS los CSV, XLSX y PARQUET dentro del proyecto
# ------------------------------------------------------------

allowed_extensions = [".csv", ".xlsx", ".xls", ".parquet"]

project_files = [
    p for p in project_path.rglob("*")
    if p.is_file() and p.suffix.lower() in allowed_extensions
]

print("\nARCHIVOS TABULARES EN TODO Nearshoring_Project")
print("=" * 100)
print(f"Total encontrados: {len(project_files)}")

for i, p in enumerate(project_files):
    print(f"{i} -> {p}")


# ------------------------------------------------------------
# 8. Mostrar archivos más recientes dentro del proyecto
# ------------------------------------------------------------

recent_files = []

for p in project_files:
    try:
        recent_files.append({
            "path": str(p),
            "name": p.name,
            "suffix": p.suffix,
            "modified_time": datetime.fromtimestamp(p.stat().st_mtime),
            "size_mb": p.stat().st_size / (1024 ** 2)
        })
    except Exception:
        pass

recent_df = pd.DataFrame(recent_files)

if len(recent_df) > 0:
    recent_df = recent_df.sort_values("modified_time", ascending=False)

    print("\nARCHIVOS TABULARES MÁS RECIENTES")
    print("=" * 100)
    display(recent_df.head(30))
else:
    print("\nNo encontré archivos tabulares dentro del proyecto.")

In [7]:
# ============================================================
# INVENTARIO COMPLETO DEL PROYECTO NEARSHORING / PORTAFOLIO DS
# Todo en una sola celda para Google Colab
# ============================================================

from pathlib import Path
import pandas as pd
import os
from google.colab import drive

# ------------------------------------------------------------
# 1. Montar Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive", force_remount=False)

# ------------------------------------------------------------
# 2. Definir ruta del proyecto
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/Nearshoring_Project")

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"No existe la ruta del proyecto: {PROJECT_DIR}\n"
        "Revisa si el nombre de la carpeta cambió o si está en otra ubicación."
    )

print("Project directory:")
print(PROJECT_DIR)

# ------------------------------------------------------------
# 3. Configuración de extensiones a inventariar
# ------------------------------------------------------------

extensions = [
    ".ipynb", ".py", ".md", ".txt",
    ".csv", ".xlsx", ".xls",
    ".geojson", ".json",
    ".shp", ".dbf", ".shx", ".prj", ".cpg",
    ".html",
    ".png", ".jpg", ".jpeg", ".svg",
    ".pdf"
]

# ------------------------------------------------------------
# 4. Crear inventario completo
# ------------------------------------------------------------

files = []

for ext in extensions:
    for p in PROJECT_DIR.rglob(f"*{ext}"):
        try:
            size_mb = round(p.stat().st_size / (1024 * 1024), 3)
        except OSError:
            size_mb = None

        files.append({
            "file_name": p.name,
            "extension": p.suffix.lower(),
            "relative_path": str(p.relative_to(PROJECT_DIR)),
            "folder": str(p.parent.relative_to(PROJECT_DIR)),
            "size_mb": size_mb
        })

inventory = (
    pd.DataFrame(files)
    .sort_values(["extension", "folder", "file_name"])
    .reset_index(drop=True)
)

if inventory.empty:
    raise ValueError("No se encontraron archivos con las extensiones configuradas.")

# ------------------------------------------------------------
# 5. Clasificación sugerida de archivos
# ------------------------------------------------------------

def classify_file(row):
    path = row["relative_path"].lower()
    ext = row["extension"].lower()
    name = row["file_name"].lower()

    if ext == ".ipynb":
        return "notebook"
    elif ext == ".py":
        return "script"
    elif ext in [".md", ".txt"]:
        return "documentation"
    elif ext in [".png", ".jpg", ".jpeg", ".svg", ".html"]:
        return "visual_output"
    elif ext in [".geojson", ".shp", ".dbf", ".shx", ".prj", ".cpg"]:
        return "geo_data"
    elif ext in [".csv", ".xlsx", ".xls"]:
        if "raw" in path or "inegi_data" in path or "denue" in path and "processed" not in path:
            return "source_or_raw_data"
        elif "processed" in path:
            return "processed_data"
        else:
            return "data_other"
    elif ext == ".pdf":
        return "pdf_document"
    else:
        return "other"

inventory["suggested_type"] = inventory.apply(classify_file, axis=1)

# ------------------------------------------------------------
# 6. Resúmenes útiles
# ------------------------------------------------------------

summary_by_extension = (
    inventory
    .groupby("extension", dropna=False)
    .agg(
        n_files=("file_name", "count"),
        total_mb=("size_mb", "sum")
    )
    .reset_index()
    .sort_values("total_mb", ascending=False)
)

summary_by_type = (
    inventory
    .groupby("suggested_type", dropna=False)
    .agg(
        n_files=("file_name", "count"),
        total_mb=("size_mb", "sum")
    )
    .reset_index()
    .sort_values("total_mb", ascending=False)
)

summary_by_folder = (
    inventory
    .groupby("folder", dropna=False)
    .agg(
        n_files=("file_name", "count"),
        total_mb=("size_mb", "sum")
    )
    .reset_index()
    .sort_values("total_mb", ascending=False)
)

notebooks = (
    inventory[inventory["extension"] == ".ipynb"]
    .sort_values(["folder", "file_name"])
    .reset_index(drop=True)
)

datasets = (
    inventory[inventory["extension"].isin([".csv", ".xlsx", ".xls"])]
    .sort_values(["folder", "file_name"])
    .reset_index(drop=True)
)

geo_files = (
    inventory[inventory["extension"].isin([".geojson", ".shp", ".dbf", ".shx", ".prj", ".cpg"])]
    .sort_values(["folder", "file_name"])
    .reset_index(drop=True)
)

visual_outputs = (
    inventory[inventory["extension"].isin([".html", ".png", ".jpg", ".jpeg", ".svg"])]
    .sort_values(["folder", "file_name"])
    .reset_index(drop=True)
)

largest_files = (
    inventory
    .sort_values("size_mb", ascending=False)
    .head(25)
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 7. Guardar outputs en carpeta de documentación del proyecto
# ------------------------------------------------------------

DOCS_DIR = PROJECT_DIR / "project_audit"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

inventory_path = DOCS_DIR / "project_inventory_full.csv"
summary_extension_path = DOCS_DIR / "summary_by_extension.csv"
summary_type_path = DOCS_DIR / "summary_by_type.csv"
summary_folder_path = DOCS_DIR / "summary_by_folder.csv"
notebooks_path = DOCS_DIR / "notebooks_inventory.csv"
datasets_path = DOCS_DIR / "datasets_inventory.csv"
geo_path = DOCS_DIR / "geo_files_inventory.csv"
visual_path = DOCS_DIR / "visual_outputs_inventory.csv"
largest_path = DOCS_DIR / "largest_files.csv"

inventory.to_csv(inventory_path, index=False, encoding="utf-8-sig")
summary_by_extension.to_csv(summary_extension_path, index=False, encoding="utf-8-sig")
summary_by_type.to_csv(summary_type_path, index=False, encoding="utf-8-sig")
summary_by_folder.to_csv(summary_folder_path, index=False, encoding="utf-8-sig")
notebooks.to_csv(notebooks_path, index=False, encoding="utf-8-sig")
datasets.to_csv(datasets_path, index=False, encoding="utf-8-sig")
geo_files.to_csv(geo_path, index=False, encoding="utf-8-sig")
visual_outputs.to_csv(visual_path, index=False, encoding="utf-8-sig")
largest_files.to_csv(largest_path, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 8. Mostrar resultados sin truncar demasiado
# ------------------------------------------------------------

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 140)

print("\n============================================================")
print("INVENTARIO GENERADO")
print("============================================================")
print(f"Total de archivos inventariados: {len(inventory)}")
print(f"Carpeta de auditoría creada en:\n{DOCS_DIR}")

print("\nArchivos guardados:")
print(f"- {inventory_path}")
print(f"- {summary_extension_path}")
print(f"- {summary_type_path}")
print(f"- {summary_folder_path}")
print(f"- {notebooks_path}")
print(f"- {datasets_path}")
print(f"- {geo_path}")
print(f"- {visual_path}")
print(f"- {largest_path}")

print("\n============================================================")
print("RESUMEN POR TIPO SUGERIDO")
print("============================================================")
display(summary_by_type)

print("\n============================================================")
print("RESUMEN POR EXTENSIÓN")
print("============================================================")
display(summary_by_extension)

print("\n============================================================")
print("NOTEBOOKS")
print("============================================================")
display(notebooks[["file_name", "relative_path", "folder", "size_mb"]])

print("\n============================================================")
print("DATASETS CSV / EXCEL")
print("============================================================")
display(datasets[["file_name", "extension", "relative_path", "folder", "size_mb", "suggested_type"]])

print("\n============================================================")
print("MAPAS / ARCHIVOS GEO")
print("============================================================")
display(geo_files[["file_name", "extension", "relative_path", "folder", "size_mb"]])

print("\n============================================================")
print("OUTPUTS VISUALES: HTML / IMÁGENES")
print("============================================================")
display(visual_outputs[["file_name", "extension", "relative_path", "folder", "size_mb"]])

print("\n============================================================")
print("TOP 25 ARCHIVOS MÁS PESADOS")
print("============================================================")
display(largest_files[["file_name", "extension", "relative_path", "folder", "size_mb", "suggested_type"]])

print("\n============================================================")
print("INVENTARIO COMPLETO")
print("============================================================")
display(inventory)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory:
/content/drive/MyDrive/Nearshoring_Project

INVENTARIO GENERADO
Total de archivos inventariados: 156
Carpeta de auditoría creada en:
/content/drive/MyDrive/Nearshoring_Project/project_audit

Archivos guardados:
- /content/drive/MyDrive/Nearshoring_Project/project_audit/project_inventory_full.csv
- /content/drive/MyDrive/Nearshoring_Project/project_audit/summary_by_extension.csv
- /content/drive/MyDrive/Nearshoring_Project/project_audit/summary_by_type.csv
- /content/drive/MyDrive/Nearshoring_Project/project_audit/summary_by_folder.csv
- /content/drive/MyDrive/Nearshoring_Project/project_audit/notebooks_inventory.csv
- /content/drive/MyDrive/Nearshoring_Project/project_audit/datasets_inventory.csv
- /content/drive/MyDrive/Nearshoring_Project/project_audit/geo_files_inventory.csv
- /content/drive/MyDrive/Nearshoring_Project/project_audit/visu

,suggested_type,n_files,total_mb
3,processed_data,54,827.895
4,source_or_raw_data,34,733.975
2,geo_data,25,422.408
0,data_other,20,7.633
5,visual_output,15,6.034
1,documentation,8,0.048



RESUMEN POR EXTENSIÓN


,extension,n_files,total_mb
1,.csv,108,1569.503
6,.shp,5,364.424
2,.dbf,5,54.706
4,.png,15,6.034
7,.shx,5,3.276
3,.md,5,0.043
8,.txt,3,0.005
5,.prj,5,0.002
0,.cpg,5,0.000



NOTEBOOKS


...

,file_name,extension,relative_path,folder,size_mb
0,01_opportunity_category_counts_2023.png,.png,figures/executive_outputs/counts/01_opportunity_category_counts_2023.png,figures/executive_outputs/counts,0.142
1,02_opportunity_subtype_counts_2023.png,.png,figures/executive_outputs/counts/02_opportunity_subtype_counts_2023.png,figures/executive_outputs/counts,0.154
2,03_scaled_b2b_supply_status_2018_2023.png,.png,figures/executive_outputs/counts/03_scaled_b2b_supply_status_2018_2023.png,figures/executive_outputs/counts,0.109
3,04_priority_opportunity_status_2018_2023.png,.png,figures/executive_outputs/counts/04_priority_opportunity_status_2018_2023.png,figures/executive_outputs/counts,0.103
4,05_atomized_b2b_opportunity_status_2018_2023.png,.png,figures/executive_outputs/counts/05_atomized_b2b_opportunity_status_2018_2023.png,figures/executive_outputs/counts,0.099
5,06_opportunity_score_distribution_2023.png,.png,figures/executive_outputs/counts/06_opportunity_score_distribution_2023.png,figures/executive_outputs/counts,0.093
6,07_industrial_demand_vs_scaled_b2b_supply_2023.png,.png,figures/executive_outputs/counts/07_industrial_demand_vs_scaled_b2b_supply_2023.png,figures/executive_outputs/counts,0.398
7,08_general_b2b_supply_vs_scaled_b2b_supply_2023.png,.png,figures/executive_outputs/counts/08_general_b2b_supply_vs_scaled_b2b_supply_2023.png,figures/executive_outputs/counts,0.387
8,map_denue_b2b_medium_large_2023.png,.png,outputs/figures/map_denue_b2b_medium_large_2023.png,outputs/figures,1.143
9,map_denue_b2b_support_2023.png,.png,outputs/figures/map_denue_b2b_support_2023.png,outputs/figures,1.142



TOP 25 ARCHIVOS MÁS PESADOS


,file_name,extension,relative_path,folder,size_mb,suggested_type
0,denue_industrial_support_base.csv,.csv,data/processed/denue_industrial_support_base.csv,data/processed,785.370,processed_data
1,denue_31_33_2023.csv,.csv,data/raw/DNUE/2023/denue_31_33_2023.csv,data/raw/DNUE/2023,238.427,source_or_raw_data
2,denue_31_33_2018.csv,.csv,data/raw/DNUE/2018/denue_31_33_2018.csv,data/raw/DNUE/2018,202.005,source_or_raw_data
3,00a.shp,.shp,data/Geo/conjunto_de_datos/00a.shp,data/Geo/conjunto_de_datos,200.409,geo_data
4,00l.shp,.shp,data/Geo/conjunto_de_datos/00l.shp,data/Geo/conjunto_de_datos,82.587,geo_data
5,SAIC_INEGI.csv,.csv,Inegi_data/SAIC_INEGI.csv,Inegi_data,64.750,source_or_raw_data
6,00mun.shp,.shp,data/Geo/conjunto_de_datos/00mun.shp,data/Geo/conjunto_de_datos,58.079,geo_data
7,denue_54_2023.csv,.csv,data/raw/DNUE/2023/denue_54_2023.csv,data/raw/DNUE/2023,55.938,source_or_raw_data
8,denue_56_2018.csv,.csv,data/raw/DNUE/2018/denue_56_2018.csv,data/raw/DNUE/2018,50.314,source_or_raw_data
9,00lpr.dbf,.dbf,data/Geo/conjunto_de_datos/00lpr.dbf,data/Geo/conjunto_de_datos,44.886,geo_data



INVENTARIO COMPLETO


,file_name,extension,relative_path,folder,size_mb,suggested_type
0,00a.cpg,.cpg,data/Geo/conjunto_de_datos/00a.cpg,data/Geo/conjunto_de_datos,0.000,geo_data
1,00ent.cpg,.cpg,data/Geo/conjunto_de_datos/00ent.cpg,data/Geo/conjunto_de_datos,0.000,geo_data
2,00l.cpg,.cpg,data/Geo/conjunto_de_datos/00l.cpg,data/Geo/conjunto_de_datos,0.000,geo_data
3,00lpr.cpg,.cpg,data/Geo/conjunto_de_datos/00lpr.cpg,data/Geo/conjunto_de_datos,0.000,geo_data
4,00mun.cpg,.cpg,data/Geo/conjunto_de_datos/00mun.cpg,data/Geo/conjunto_de_datos,0.000,geo_data
5,project_inventory.csv,.csv,project_inventory.csv,.,0.018,data_other
6,SAIC_Exporta_202656_10535819.csv,.csv,Inegi_data/SAIC_Exporta_202656_10535819.csv,Inegi_data,0.254,source_or_raw_data
7,SAIC_INEGI.csv,.csv,Inegi_data/SAIC_INEGI.csv,Inegi_data,64.750,source_or_raw_data
8,denue_industrial_support_base.csv,.csv,data/processed/denue_industrial_support_base.csv,data/processed,785.370,processed_data
9,municipality_denue_support_summary.csv,.csv,data/processed/municipality_denue_support_summary.csv,data/processed,0.889,processed_data


In [8]:
# ============================================================
# BUSCAR NOTEBOOKS .ipynb EN TODO GOOGLE DRIVE
# Y DETECTAR POSIBLES NOTEBOOKS DEL PROYECTO NEARSHORING
# ============================================================

from pathlib import Path
import pandas as pd
from google.colab import drive

# ------------------------------------------------------------
# 1. Montar Drive
# ------------------------------------------------------------

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE_ROOT / "Nearshoring_Project"

if not DRIVE_ROOT.exists():
    raise FileNotFoundError("No se encontró Google Drive montado en /content/drive/MyDrive")

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"No existe PROJECT_DIR: {PROJECT_DIR}")

# ------------------------------------------------------------
# 2. Buscar todos los notebooks en Drive
# ------------------------------------------------------------

notebook_files = []

for p in DRIVE_ROOT.rglob("*.ipynb"):
    try:
        size_mb = round(p.stat().st_size / (1024 * 1024), 3)
        modified_time = pd.to_datetime(p.stat().st_mtime, unit="s")
    except OSError:
        size_mb = None
        modified_time = None

    relative_path = str(p.relative_to(DRIVE_ROOT))

    notebook_files.append({
        "file_name": p.name,
        "relative_path_from_drive": relative_path,
        "full_path": str(p),
        "folder": str(p.parent.relative_to(DRIVE_ROOT)),
        "size_mb": size_mb,
        "modified_time": modified_time
    })

notebooks = (
    pd.DataFrame(notebook_files)
    .sort_values("modified_time", ascending=False)
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Detectar notebooks posiblemente relacionados con el proyecto
# ------------------------------------------------------------

keywords = [
    "nearshoring", "denue", "inegi", "saic", "b2b", "industrial",
    "manufacturing", "municipality", "municipal", "opportunity",
    "score", "scoring", "geo", "map", "mapa", "portafolio"
]

def detect_project_match(row):
    text = f"{row['file_name']} {row['relative_path_from_drive']}".lower()
    hits = [kw for kw in keywords if kw in text]
    return ", ".join(hits)

notebooks["keyword_hits"] = notebooks.apply(detect_project_match, axis=1)
notebooks["possible_project_notebook"] = notebooks["keyword_hits"].str.len() > 0

project_candidates = (
    notebooks[notebooks["possible_project_notebook"]]
    .copy()
    .sort_values(["modified_time", "size_mb"], ascending=[False, False])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Guardar inventarios
# ------------------------------------------------------------

AUDIT_DIR = PROJECT_DIR / "project_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

all_notebooks_path = AUDIT_DIR / "all_drive_notebooks_inventory.csv"
candidate_notebooks_path = AUDIT_DIR / "possible_project_notebooks_inventory.csv"

notebooks.to_csv(all_notebooks_path, index=False, encoding="utf-8-sig")
project_candidates.to_csv(candidate_notebooks_path, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 5. Mostrar resultados
# ------------------------------------------------------------

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 160)

print("============================================================")
print("BÚSQUEDA DE NOTEBOOKS COMPLETADA")
print("============================================================")
print(f"Total de notebooks encontrados en Drive: {len(notebooks)}")
print(f"Posibles notebooks del proyecto: {len(project_candidates)}")

print("\nArchivos guardados:")
print(f"- {all_notebooks_path}")
print(f"- {candidate_notebooks_path}")

print("\n============================================================")
print("POSIBLES NOTEBOOKS DEL PROYECTO")
print("============================================================")
display(project_candidates[[
    "file_name",
    "relative_path_from_drive",
    "size_mb",
    "modified_time",
    "keyword_hits"
]])

print("\n============================================================")
print("TODOS LOS NOTEBOOKS EN DRIVE")
print("============================================================")
display(notebooks[[
    "file_name",
    "relative_path_from_drive",
    "size_mb",
    "modified_time",
    "keyword_hits"
]])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BÚSQUEDA DE NOTEBOOKS COMPLETADA
Total de notebooks encontrados en Drive: 12
Posibles notebooks del proyecto: 4

Archivos guardados:
- /content/drive/MyDrive/Nearshoring_Project/project_audit/all_drive_notebooks_inventory.csv
- /content/drive/MyDrive/Nearshoring_Project/project_audit/possible_project_notebooks_inventory.csv

POSIBLES NOTEBOOKS DEL PROYECTO


,file_name,relative_path_from_drive,size_mb,modified_time,keyword_hits
0,Final_Scores.ipynb,Final_Scores.ipynb,2.178,2026-06-24 14:37:28,score
1,DENUE.ipynb,Colab Notebooks/DENUE.ipynb,0.692,2026-05-20 19:43:54,denue
2,INEGI_SAIC.ipynb,Colab Notebooks/INEGI_SAIC.ipynb,0.996,2026-05-13 02:07:42,"inegi, saic"
3,Readme_Nearshoring.ipynb,Colab Notebooks/Readme_Nearshoring.ipynb,0.035,2026-05-12 19:22:43,nearshoring



TODOS LOS NOTEBOOKS EN DRIVE


,file_name,relative_path_from_drive,size_mb,modified_time,keyword_hits
0,Data_Merger.ipynb,Data_Merger.ipynb,0.541,2026-07-04 12:42:24,
1,WC26_2.ipynb,Colab Notebooks/WC26_2.ipynb,0.013,2026-06-24 17:13:36,
2,Final_Scores.ipynb,Final_Scores.ipynb,2.178,2026-06-24 14:37:28,score
3,Elo_WC26.ipynb,Colab Notebooks/Elo_WC26.ipynb,0.133,2026-06-12 04:57:13,
4,Stats_Iniciales.ipynb,Colab Notebooks/Stats_Iniciales.ipynb,0.608,2026-06-08 23:14:44,
5,DENUE.ipynb,Colab Notebooks/DENUE.ipynb,0.692,2026-05-20 19:43:54,denue
6,INEGI_SAIC.ipynb,Colab Notebooks/INEGI_SAIC.ipynb,0.996,2026-05-13 02:07:42,"inegi, saic"
7,Readme_Nearshoring.ipynb,Colab Notebooks/Readme_Nearshoring.ipynb,0.035,2026-05-12 19:22:43,nearshoring
8,Untitled0.ipynb,Colab Notebooks/Untitled0.ipynb,0.000,2026-05-12 19:20:20,
9,01_data_discovery.ipynb,Colab Notebooks/01_data_discovery.ipynb,0.000,2026-05-06 16:37:43,
